<div style="
  background-color:#ffffff;
  color:#1f2937;
  padding:28px 24px;
  border-radius:14px;
  border:1px solid #d1d5db;
  font-family:'Segoe UI', Arial, sans-serif;
  text-align:center;
  box-shadow:0 6px 18px rgba(0,0,0,0.06);
">

  <img src="../Su.svg.png" alt="Sorbonne Université Logo" style="
    width:95px;
    height:auto;
    margin-bottom:16px;
  ">

  <div style="
    display:inline-block;
    font-size:22px;
    font-weight:700;
    color:#111827;
    padding-bottom:8px;
    margin-bottom:16px;
  ">
    A-Abo
  </div>

  <div style="font-size:18px; font-weight:600; color:#374151;">
    Projet académique de bases de données
  </div>

  <div style="font-size:15px; margin-top:6px; color:#1e3a8a; font-weight:600;">
    Sorbonne Université — Année universitaire 2025–2026
  </div>

  <div style="
    margin:18px auto 0 auto;
    max-width:650px;
    font-size:12px;
    line-height:1.5;
    color:#6b7280;
    font-style:italic;
  ">
    Ce document est fourni à titre pédagogique et peut faire l’objet de corrections ou d’améliorations.
  </div>

</div>

<center> <h1> TD 7 et TME7 :  REQUÊTES D'AGRÉGATION ET DIVISION

<h5> Ce TD/TME utilise les données contenues dans les fichiers **Base Astronomie_H2.sql** et **bd-jo-v2_H2.sql**

# Installation H2 

Le système utilisé pendant les TME est H2.

In [1]:
pip install psycopg2-binary

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


Télécharger le pilote de H2

**Relancez le kernel**: Kernel -> Restart kernel ...

https://www.psycopg.org/docs/
http://localhost:8082

In [2]:
import psycopg2
import os
local_dir = "./data"
os.makedirs(local_dir, exist_ok=True)
os.listdir(local_dir)

[]

**Attention**: vous pouvez cliquer sur les 3 lignes dans la fenêtre de gauche pour d'afficher les différentes sections du notebook  

# Démarrage du serveur H2
Démarrer un serveur de base de données H2 en arrière-plan, accessible uniquement localement(localhost, 127.0.0.1) sur le port 8082, avec les fichiers des bases de données stockés dans ./data (un fichier *.mv.db par base de données)

In [3]:
port = 8082
print(f'Le numero du port utilisé pour la connexion à la BD est: {port}')

Le numero du port utilisé pour la connexion à la BD est: 8082


In [4]:
%%bash --bg --out output -s "$port"
java -Dh2.bindAddress=127.0.0.1 -cp h2-2.1.214.jar:postgresql-42.6.0.jar org.h2.tools.Server -pg -pgPort $1 -baseDir ./data -ifNotExists

Tester si le serveur a été démarré

In [5]:
%%bash
nc -zv 127.0.0.1 8082

Connection to 127.0.0.1 8082 port [tcp/*] succeeded!


# Connexion du client au serveur H2
Se connecter au serveur H2 précédemment lancé en utilisant un nom de base de données, un login et un mot de passe. Si une connexion existante existe, elle est fermée avant d’en créer une nouvelle. Si la base n’existe pas encore, H2 la crée automatiquement dans le répertoire de données ./data (fichier *.mv.db). La fonction retourne un objet de connexion utilisable pour exécuter des commandes SQL sur le serveur.

In [6]:
def connect_H2(db,user,password):
    global connection
    try:
        connection
    except:
        connection = None
    if connection != None:
        try:
            connection.close()
            print("Connection closed")
        except  Error as e:
            print(f"The error '{e}' occurred")
    try:
        # connection = sqlite3.connect(path,isolation_level='DEFERRED')
        connection = psycopg2.connect(f"dbname={db} user={user} password={password} host=localhost port={port}")
        print("Connection to H2 DB successful")
    except Exception as e:
        print(f"The error '{e}' occurred")
    return connection

In [7]:
def connect_H2(db, user, password):
    global connection
    try:
        connection
    except:
        connection = None

    if connection != None:
        try:
            connection.close()
            print("Connection closed")
        except Exception as e:
            print(f"The error '{e}' occurred")

    try:
        connection = psycopg2.connect(
            f"dbname={db} user={user} password={password} host=127.0.0.1 port={port} gssencmode=disable sslmode=disable"
        )
        print("Connection to H2 DB successful")
    except Exception as e:
        print(f"The error '{e}' occurred")

    return connection

In [8]:
#Connexion à la base H2 nommée "jo<port>" avec l’utilisateur et le mot de passe spécifiés. Si la base n’existe pas encore, H2 la crée automatiquement. 
#Toute connexion précédente est fermée avant d’en établir une nouvelle.
connection = connect_H2("jo"+f'{port}',"user","pass")

Connection to H2 DB successful


# Fonction execute

Exécute une commande SQL sur la connexion fournie, affiche le nombre de lignes affectées et, si demandé, présente les résultats sous forme de tableau lisible avec les noms de colonnes. Le curseur peut être fermé automatiquement après l’exécution, et toute erreur est affichée. La fonction retourne le curseur utilisé pour accéder aux résultats ou None en cas d’échec.

In [9]:
def execute(connection, query, show=True, close=True, top=10):
    try:
        cursor = connection.cursor()
        cursor.execute(query)
        if cursor.rowcount >0:
          print(cursor.rowcount,"rows,", f"display only {top} rows")
        else:
          print(cursor.rowcount,"rows")
        if show and cursor.rowcount:
            names = [desc[0] for desc in cursor.description]

            lengths={}
            for attr in names:
                lengths[attr]=len(attr)
            for ligne in cursor:
                i=0
                for attr in ligne:
                    lengths[names[i]]=max(lengths[names[i]],len(str(attr)))
                    i=i+1
            print('|',end='')
            for attr in names:
                print(str(attr).ljust(lengths[attr]),end='|')
            print()
            print('|',end='')
            for attr in names:
                print(''.ljust(lengths[attr]+1,'-'),end='')
            print()
            cursor.execute(query)
            n=0
            for ligne in cursor:
                i=0
                print('|',end='')
                for attr in ligne:
                    print(str(attr)[:lengths[names[i]]].ljust(lengths[names[i]]),end='|')
                    i+=1
                print()
                n+=1
                if n>top :
                  break
        if close:
            cursor.close()
    except Exception as e:
        print(f"The error '{e}' occurred")
        cursor=None
    return cursor
print('fonction définie')

fonction définie


# Affichage du schéma d'une table

In [10]:
def show_table(connection,table_name):

    print(f'*********** Attributs de la table : {table_name} ************')
    query=f"""
        SELECT table_name,
              column_name,
              data_type,
              column_default,
              is_nullable,
              character_maximum_length,
              numeric_precision
        FROM information_schema.columns
        where lower(table_name)  = '{table_name}'
        """
    execute(connection,query)


    print(f'\n*********** Contraintes de la table : {table_name} ************')
    query = f"""
        SELECT tc.constraint_name, 
               tc.constraint_type, 
               cc.check_clause
        FROM information_schema.table_constraints tc
        LEFT JOIN information_schema.check_constraints cc 
               ON tc.constraint_name = cc.constraint_name
        WHERE lower(tc.table_name) = '{table_name}'
        AND tc.table_schema = 'public'
        ORDER BY tc.constraint_type
        """
    execute(connection, query)

# Affichage du schéma global 

In [11]:
def show_schema(connection):

    print('*********** Tables ************')
    query="""
    select TABLE_NAME
    from INFORMATION_SCHEMA.TABLES
    where TABLE_SCHEMA = 'public'
    """
    execute(connection,query,show=True,top=1000)

    print('\n\n*********** Domaines ************')
    query="""
    SELECT domain_name,check_clause
    FROM information_schema.domain_constraints a left outer join
         information_schema.check_constraints b
         on a.constraint_name=b.constraint_name
    """
    execute(connection,query,top=1000)

    print('\n\n*********** Attributs ************')
    query=f"""
    SELECT c.table_name,
           c.column_name,
           c.data_type,
           c.column_default,
           c.is_nullable,
           c.character_maximum_length,
           c.numeric_precision
    FROM INFORMATION_SCHEMA.TABLES t, information_schema.columns c
    where t.table_name=c.table_name
    and t.TABLE_SCHEMA = 'public'
    """
    execute(connection,query,top=1000)

    print('\n\n*********** Contraintes d''Intégrité ************')
    query="""
    SELECT 
        tc.table_name, 
        tc.constraint_name, 
        tc.constraint_type,
        cc.check_clause
    FROM information_schema.table_constraints tc
    LEFT JOIN information_schema.check_constraints cc 
        ON tc.constraint_name = cc.constraint_name
    WHERE tc.table_schema = 'public'
    ORDER BY tc.table_name, tc.constraint_type
    """
    execute(connection,query,top=1000)

# TD7:  REQUÊTES D'AGRÉGATION ET DIVISION

On veut réaliser une application pour des données astronomiques. Les observations, les dates et les
valeurs observées sont inventées pour les besoins de l’exercice. Le rayon des astres est en km. On
suppose que les attributs ne peuvent pas être null. On considère le schéma suivant :


**Categorie**(<u>idC</u>,nom)<br/>
**Astre**(<u>idA</u>,nom,rayon,idC*) avec idC référence Categorie(idC)<br/>
**TourneAutour**(<u>idA1*</u>,idA2*,position) avec idA1 et idA2 référence Astre(idA)<br/>
**Observation**(<u>idO</u>,idA*,dateObs,valObs) avec idA référence Astre(idA)<br/>

Exemples de données :
 - La Terre est la troisième planète qui tourne autour du Soleil.<br/>
 - Le 10 mai 2010, on a réalisé deux observations du Soleil avec les valeurs 12001 et 12003.


## Créer les tables et charger les données Base Astronomie_H2.sql

In [12]:
schemafile=open("Base Astronomie_H2.sql",mode="r",encoding='utf-8')
create_schema=schemafile.read()
execute(connection,"drop all objects")
execute(connection, create_schema)
connection.commit()

0 rows
0 rows


<u>**B - Requêtes: Fonctions d’agrégation « COUNT, SUM, AVG, MIN, MAX »**</u>

## **Q5.**. 
Le nombre de catégories. Résultat : 3

In [13]:
query="""
select count(*)
from categorie
"""
execute(connection,query,show=True)

1 rows, display only 10 rows
|?column?|
|---------
|3       |


<cursor object at 0x7f81b83fa6b0; closed: -1>

 ## **Q6.** 
Le nombre de catégories pour lesquelles on connaît au moins un astre. Résultat : 3

In [14]:
query="""
select count(*)
from categorie c
where c.idc = any (select a.idc 
                    from astre a)
"""
execute(connection,query,show=True)

1 rows, display only 10 rows
|?column?|
|---------
|3       |


<cursor object at 0x7f81b83fa890; closed: -1>

In [15]:
query="""
select count(*)
from categorie c
where exists (select *
                from astre a
                where a.idc = c.idc)
"""
execute(connection,query,show=True)

1 rows, display only 10 rows
|?column?|
|---------
|3       |


<cursor object at 0x7f81b83fa7a0; closed: -1>

##  **Q7.**) 
Le rayon minimal et le rayon maximal de tous les astres. Résultat : (1737, 696342)

In [16]:
query="""
select min(rayon) , max(rayon) 
from astre
"""
execute(connection,query,show=True)

1 rows, display only 10 rows
|?column?|?column?|
|------------------
|1737    |696342  |


<cursor object at 0x7f81b83fa980; closed: -1>

## **Q8.** 
Le rayon moyen en milliers de km (et non pas en km), arrondi à 2 chiffres après la virgule.
Résultat : ‘176,96 milliers km’. 

Aide : utilisez :

a) round(valeur,nb) pour garder seulement nb décimales à valeur

b) l’opérateur de concaténation || pour concaténer des chaînes, attributs… :
select ‘abc’ || attribut || ‘def’ from UneTable ;

In [17]:
query="""
select round(avg(rayon),2) || ' milliers km.' as YOUR_TITLE_GOES_HERE_OPTIONAL
from astre
"""
execute(connection,query,show=True)

1 rows, display only 10 rows
|your_title_goes_here_optional|
|------------------------------
|176960.0 milliers km.        |


<cursor object at 0x7f81b83faa70; closed: -1>

## **Q9.a)** 
Le rayon de l'astre de rayon maximal. Résultat : 696342

In [18]:
query="""
select max(rayon) as MAXX
from astre

"""
execute(connection,query,show=True)

1 rows, display only 10 rows
|maxx  |
|-------
|696342|


<cursor object at 0x7f81b83fac50; closed: -1>

##  **Q9.b)** 
Le nom et le rayon de l'astre de rayon maximal. Résultat : (Soleil, 696342)

In [42]:
query="""
select nom, rayon
from astre
where rayon in (select max(a.rayon)
                    from astre a)



"""
execute(connection,query,show=True)

1 rows, display only 10 rows
|nom   |rayon |
|--------------
|Soleil|696342|


<cursor object at 0x71d4581e4220; closed: -1>

In [45]:
query="""
select a.nom, a.rayon
from astre a
where not exists ( select *
                    from astre a2
                    where a.rayon < a2.rayon)



"""
execute(connection,query,show=True)

1 rows, display only 10 rows
|nom   |rayon |
|--------------
|Soleil|696342|


<cursor object at 0x71d4581e4130; closed: -1>

<u>**C - Requêtes: Partitionnement « group by »**</u>

##  **Q10.** 
Pour chaque catégorie, l'identifiant de la catégorie et le nombre d'astres de cette catégorie
(une seule table suffit), ordonner par nombre d’astres décroissant.
Résultats (3 lignes) : (11, 2) ; (10,1) ; (12,1)

In [48]:
query="""
select idc , count(*)
from astre 
group by idc
order by count(*) desc
"""
execute(connection,query,show=True)

3 rows, display only 10 rows
|idc|?column?|
|-------------
|11 |2       |
|10 |1       |
|12 |1       |


<cursor object at 0x71d4581e44f0; closed: -1>

##  **Q11.a)** 
Pour chaque astre, l’idA de l’astre et le nombre d'astres qui lui tournent autour. Pour
simplifier, prendre que les astres qui ont au moins un autre astre qui leur tourne autour.
Résultats (2 lignes) : (100, 2) ; (101,1).

In [53]:
query="""
select ida2 , count(*)
from tourneautour
where ida1 is not null
group by ida2


"""
execute(connection,query,show=True)

2 rows, display only 10 rows
|ida2|?column?|
|--------------
|100 |2       |
|101 |1       |


<cursor object at 0x71d4581e46d0; closed: -1>

##  **Q11. b)** 
Même question, mais on veut le nom de l’astre au lieu de l’idA.
Résultats (2 lignes) : (Soleil, 2) ; (Terre, 1) 

In [58]:
query="""
select a.nom , count(*)
from astre a, tourneautour t
where t.ida2 = a.ida
group by a.ida , a.nom
"""
execute(connection,query,show=True)


2 rows, display only 10 rows
|nom   |?column?|
|----------------
|Soleil|2       |
|Terre |1       |


<cursor object at 0x71d4581e4c70; closed: -1>

##  **Q12.** 
Pour chaque couple (astre,date), le nom de l'astre, la date de l'observation et la moyenne des
valeurs des observations réalisées.
Résultats : (Soleil,2010-05-10,12002) ; (Terre,2013-12-18,8005) ; (Lune,2014-08-27)

In [72]:
query="""
select a.nom ,o.dateobs , avg(o.valobs)
from observation o , astre a
where a.ida = o.ida
group by o.dateobs , a.ida
"""
execute(connection,query,show=True)

3 rows, display only 10 rows
|nom   |dateobs   |?column?|
|---------------------------
|Soleil|2010-05-10|12002.0 |
|Terre |2013-12-18|8005.0  |
|Lune  |2014-08-27|3007.0  |


<cursor object at 0x71d4581e56c0; closed: -1>

##  **Q13.a)**  
Pour chaque catégorie, l'idC de la catégorie et le rayon maximal des astres de cette
catégorie, ordonner par rayon croissant. Aide : une seule table suffit.
Résultats (3 lignes) : (12,1737) ; (11,6371) ; (10,696342)

In [75]:
query="""
select idc,max(rayon)
from astre
group by idc
order by max(rayon) asc
"""
execute(connection,query,show=True)

3 rows, display only 10 rows
|idc|?column?|
|-------------
|12 |1737    |
|11 |6371    |
|10 |696342  |


<cursor object at 0x71d4581e58a0; closed: -1>

##  **Q13.b)**  
On veut maintenant afficher non pas l’idC, mais le nom de la catégorie.
Pour chaque catégorie, le nom de la catégorie et le rayon maximal des astres de cette
catégorie, ordonner par nombre d’astres dans chaque catégorie décroissant.
Résultats (3 lignes) : (planète, 6371) ; (étoile,696342) ; (satellite, 1737)

In [78]:
query="""
select c.nom , max(a.rayon)
from astre a, categorie c
where a.idc = c.idc 
group by c.idc
order by count(*) desc

"""
execute(connection,query,show=True)

3 rows, display only 10 rows
|nom      |?column?|
|-------------------
|planète  |6371    |
|étoile   |696342  |
|satellite|1737    |


<cursor object at 0x71d4581e5a80; closed: -1>

##  **Q13.c)**  
On veut maintenant afficher non pas le nom de la catégorie, mais le nom de l’astre.
Pour chaque catégorie, l'idC de la catégorie, le(s) nom(s) de(s) astre(s) de rayon
maximal et le rayon maximal des astres de cette catégorie. Aide : deux tables suffisent.
Résultats (3 lignes) : (10, Soleil,696342) ; (11,Terre,6371) ; (12, Lune, 1737)

In [101]:
query="""
select a.idc , a.nom ,a.rayon
from astre a 
where a.rayon = (select max(a2.rayon)
                from astre a2
                where a2.idc = a.idc )

"""
execute(connection,query,show=True)

3 rows, display only 10 rows
|idc|nom   |rayon |
|------------------
|10 |Soleil|696342|
|11 |Terre |6371  |
|12 |Lune  |1737  |


<cursor object at 0x71d4581e65c0; closed: -1>

<u>**D - Requêtes: Partitionnement avec sélection de partition « group by / having »**</u>

## **Q14.** 
L'idC des catégories pour lesquelles il existe au moins deux astres dans cette catégorie.
Résultat : 11

In [103]:
query="""
select a.idc 
from astre a
group by a.idc 
having count(*) >=2
"""
execute(connection,query,show=True)

1 rows, display only 10 rows
|idc|
|----
|11 |


<cursor object at 0x71d4581e6d40; closed: -1>

## **Q15.** 
Pour chaque date (où il y a eu au moins une observation) et où il y a eu une valeur maximale
observée supérieure à 8000, la date, le nombre d'observations effectuées ce jour là et la
valeur maximale observée. Résultats(2 lignes):(2010-05-10,2, 12003);(2013-12-18, 1, 8005)

In [125]:
query="""
select o.dateobs ,count(*),max(o.valobs) 
from observation o 
group by o.dateobs
having max(o.valobs) > 8000
"""
execute(connection,query,show=True)

2 rows, display only 10 rows
|dateobs   |?column?|?column?|
|-----------------------------
|2010-05-10|2       |12003   |
|2013-12-18|1       |8005    |


<cursor object at 0x71d4581e7b50; closed: -1>

##  **Q16.** 
 Le nom des astres pour lesquels on a fait exactement deux observations. Résultat : Soleil

In [126]:
query="""
select a.nom
from astre a, observation o 
where o.ida = a.ida 
group by a.ida 
having count(*) = 2
"""
execute(connection,query,show=True)

1 rows, display only 10 rows
|nom   |
|-------
|Soleil|


<cursor object at 0x71d4581e7d30; closed: -1>

##  **Q17.** 
Le nom des astres qui ont au moins deux astres qui leur tournent autour. Résultat : Soleil

In [128]:
query="""
select a.nom
from astre a, tourneautour t
where a.ida = t.ida2 
group by a.ida 
having count(*) >= 2
"""
execute(connection,query,show=True)

1 rows, display only 10 rows
|nom   |
|-------
|Soleil|


<cursor object at 0x71d4581f0040; closed: -1>

<u>**E - Requêtes: Réécriture double négation avec Group By et Having**</u>

##  **Q18.a)** 
 Les dates d’observations pour lesquelles il y a au moins une observation pour chaque astre existant dans la bases de données.
Résultat : aucun sur cet exemple

In [ ]:
query="""

"""
execute(connection,query,show=True)

##  **Q18.b)** 
Les dates d’observations pour lesquelles il y a au moins une observation pour chaque astre observés au moins une fois.
Résultat : aucun sur cet exemple

In [ ]:
query="""


"""
execute(connection,query,show=True)


##  **Q19.a)** 
L'idC des catégories pour lesquelles il existe au moins une observation pour chaque astre
de cette catégorie. Résultats (2 lignes) : 10 ; 12

In [ ]:
query="""


"""
execute(connection,query,show=True)

##  **Q19.b)**  
Même question mais on veut afficher non pas l’idC, mais le nom de la catégorie.
Résultats (2 lignes) : étoile ; satellite

In [ ]:
query="""


"""
execute(connection,query,show=True)

# TME7: REQUÊTES D'AGRÉGATION ET DIVISION

On reprend le schéma schéma « Jeux Olympiques d'hiver 2014 ».

**PAYS** ( <u>CODEPAYS</u>, NOMP)<br/>
Ex. ('FRA', 'France')<br/>
**SPORT** ( <u>SID</u>, NOMSP)<br/>
Ex. (1, 'Biathlon')<br/>
**EPREUVE** ( <u>EPID</u>, SID*, NOMEP, CATÉGORIE, DATEDEBUT, DATEFIN)<br/>
Ex. (10, 1, 'relais 4x7,5km', 'Hommes', 22/02/2014, 22/02/2014)<br/>
**ATHLETE** ( <u>AID</u>, NOMATH, PRENOMATH, DATENAISSANCE, CODEPAYS*)<br/>
Ex. (1000, 'SOBOLEV', 'Alexey', NULL, 'RUS')<br/>
**EQUIPE** ( <u>EQID</u>, CODEPAYS*)<br/>
Ex. (30, 'SUI')<br/>
**ATHLETESEQUIPE** ( <u>EQID*, AID*</u>)<br/>
Ex. (30, 796) : L'athlète (aid=796) a participé à l'équipe (eqid=30)<br/>
**RANGINDIVIDUEL** ( <u>EPID*, AID*</u>, RANG)<br/>
Ex. (15, 61, 1) : L'athlète (aid=61) a gagné la médaille d'or (rang=1) de l'épreuve (epid=15)<br/>
**RANGEQUIPE** ( <u>EPID*, EQID*</u>, RANG)<br/>
Ex. (10, 30, 14) : L'équipe (eqid=30) a été classée 14e à l'épreuve (epid=10)<br/>

## Créer les tables et charger les données bd-jo-v2_H2.sql

In [19]:
schemafile=open("bd-jo-v2_H2.sql",mode="r",encoding='utf-8')
create_schema=schemafile.read()
execute(connection,"drop all objects")
execute(connection, create_schema)
connection.commit()

0 rows
0 rows


<u>**Requêtes: Fonctions d’agrégation « COUNT, SUM, AVG, MIN, MAX »**</u>

##  **Q1.**  
Le nombre d’athlètes.
Résultat (1 ligne) : 2431

In [20]:
query="""
select count(*)
from athlete

"""
execute(connection,query,show=True)

1 rows, display only 10 rows
|?column?|
|---------
|2431    |


<cursor object at 0x7f81b83fad40; closed: -1>

##  **Q2.**  
Le nombre d’athlètes ayant participé à au moins une épreuve en individuel.
Résultat (1 ligne) : 1558

In [21]:
query="""
select count(*)
from athlete a
where a.aid in (select r.aid
                from rangindividuel r
                 )

"""
execute(connection,query,show=True)

1 rows, display only 10 rows
|?column?|
|---------
|1558    |


<cursor object at 0x7f81b83fab60; closed: -1>

In [22]:
query="""
select count(*)
from athlete a
where a.aid = any (select r.aid
                from rangindividuel r
                 )

"""
execute(connection,query,show=True)

1 rows, display only 10 rows
|?column?|
|---------
|1558    |


<cursor object at 0x7f81b83faf20; closed: -1>

In [23]:
query="""
select count(*)
from athlete a
where exists (select *
                from rangindividuel r
                 where r.aid = a.aid )

"""
execute(connection,query,show=True)

1 rows, display only 10 rows
|?column?|
|---------
|1558    |


<cursor object at 0x7f81b83fb2e0; closed: -1>

##  **Q3.**  
L'âge moyen des sportifs dont le code pays est 'FRA' (France) au 06/02/2014.
Résultat (1 ligne) : 26,8

Aide :

– utilisez round(valeur,nb) pour garder seulement nb décimales à valeur

In [24]:
query="""
select  round(avg( datediff(year,a.datenaissance,'2014-02-06')),1)
from athlete a
where codepays = 'FRA'


"""
execute(connection,query,show=True)

1 rows, display only 10 rows
|round|
|------
|27.3 |


<cursor object at 0x7f81b83fb3d0; closed: -1>

##  **Q4.**  
La durée moyenne, minimale et maximale des épreuves.
Résultat (1 ligne) : « Durée moyenne = 1,98 min = 1 max = 16 »

Aide : utilisez l’opérateur de concaténation ||

Attention : entre le 10/01/2014 et le 13/01/2014, il y a une durée de 4 jours (et non pas 3).

In [33]:
query="""
select round(avg(datediff( 'day',e.datedebut,e.datefin+1)),2) , min(datediff('day',e.datedebut,e.datefin+1)) , max(datediff('day',e.datedebut,e.datefin+1))
from epreuve e

"""
execute(connection,query,show=True)

1 rows, display only 10 rows
|round|?column?|?column?|
|------------------------
|1.98 |1       |16      |


<cursor object at 0x7f81b83fbb50; closed: -1>

## **Q5.** 
Le nombre moyen d'athlètes par pays, c'est-à-dire le nombre d'athlètes divisé par le nombre
de pays (ayant au moins un athlète). Résultat (1 ligne) : 27,625

In [39]:
query="""
select round( count(a.aid)*1.0/count(distinct a.codepays),3)
from athlete a

"""
execute(connection,query,show=True)

1 rows, display only 10 rows
|round |
|-------
|27.625|


<cursor object at 0x7f818ab14130; closed: -1>

<u>**Partitionnement « group by »**</u>

## **Q6.** 
Pour chaque pays, le nom du pays et le nombre d’athlètes, ordonner par nombre d’athlètes
croissant.
Résultats (88 lignes) : (PAK,1) ; (HKG, 1) ; ... ; (USA, 196) ; (CAN,221)

In [44]:
query="""
select a.codepays, count(*)
from athlete a
group by a.codepays
order by count(*) asc
"""
execute(connection,query,show=True)

88 rows, display only 10 rows
|codepays|?column?|
|------------------
|BER     |1       |
|CAY     |1       |
|HKG     |1       |
|ISV     |1       |
|IVB     |1       |
|KGZ     |1       |
|LUX     |1       |
|MEX     |1       |
|MLT     |1       |
|NEP     |1       |
|PAK     |1       |


<cursor object at 0x7f818ab144f0; closed: -1>

## **Q7.** 
Le nombre moyen d'athlètes par pays (avec group by). Aide : compter le nombre
d’athlètes dans chaque pays (ayant au moins un athlète) avec une sous-requêtes dans la clause FROM, puis faire la moyenne.
Résultat (1 ligne) : 27,625

In [89]:
query="""
select avg(t.nb)
from (select codepays , count(*) as nb
        from athlete a
        group by codepays)t
"""
execute(connection,query,show=True)

1 rows, display only 10 rows
|?column?     |
|--------------
|27.6250000000|


<cursor object at 0x7f818ab15b70; closed: -1>

## **Q8.** 
Pour chaque équipe, l’eqid de l'équipe et le nombre d'athlètes, ordonner par nombre
d’athlètes décroissant.
Résultats (296 lignes) : (164,25) ; (165,25) ; (166,25) ; ... ; (180,2) ; (181, 2) ; (182, 2)

In [111]:
query="""
select e.eqid , count(*)
from athletesequipe e
group by e.eqid
order by count(*) desc
"""
execute(connection,query,show=True)

296 rows, display only 10 rows
|eqid|?column?|
|--------------
|164 |25      |
|165 |25      |
|166 |25      |
|156 |21      |
|157 |21      |
|158 |21      |
|226 |10      |
|227 |9       |
|228 |8       |
|140 |5       |
|141 |5       |


<cursor object at 0x7f818ab165c0; closed: -1>

In [109]:
query="""
select e.eqid , count(*)
from equipe e , athlete a,  athletesequipe aq
where a.aid = aq.aid and e.eqid  = aq.eqid
group by e.eqid
order by count(*) desc
"""
execute(connection,query,show=True)

296 rows, display only 10 rows
|eqid|?column?|
|--------------
|164 |25      |
|165 |25      |
|166 |25      |
|156 |21      |
|157 |21      |
|158 |21      |
|226 |10      |
|227 |9       |
|228 |8       |
|140 |5       |
|141 |5       |


<cursor object at 0x7f818ab16200; closed: -1>

## **Q9.** 
Pour chaque catégorie, la catégorie et le nombre d'épreuves.
Résultats (3 lignes) : (Femmes,43) ; (Mixte,6) ; (Hommes,49)

In [112]:
query="""
select e.categorie , count(*)
from epreuve e
group by e.categorie
"""
execute(connection,query,show=True)

3 rows, display only 10 rows
|categorie|?column?|
|-------------------
|Femmes   |43      |
|Hommes   |49      |
|Mixte    |6       |


<cursor object at 0x7f818ab166b0; closed: -1>

## **Q10.** 
Pour chaque sport, le nom du sport et le nombre d'épreuves, ordonner par nombre d'épreuves
décroissant.

Résultats (15 lignes) : (Patinage de vitesse,12) ; (Ski de fond,12) ; ... ;(Hockey sur glace,2)

In [116]:
query="""
select s.nomsp, count(*)
from epreuve e, sport s
where e.sid = s.sid 
group by s.nomsp
order by count(*) desc
"""
execute(connection,query,show=True)

15 rows, display only 10 rows
|nomsp                               |?column?|
|----------------------------------------------
|Patinage de vitesse                 |12      |
|Ski de fond                         |12      |
|Biathlon                            |11      |
|Ski acrobatique                     |10      |
|Ski alpin                           |10      |
|Surf des neiges                     |10      |
|Patinage de vitesse sur piste courte|8       |
|Patinage artistique                 |5       |
|Luge                                |4       |
|Saut à ski                          |4       |
|Bobsleigh                           |3       |


<cursor object at 0x7f818ab16a70; closed: -1>

## **Q11.** 
Pour chaque pays, le code du pays, le nombre de médailles en épreuve individuelle gagnées
et le nombre d'athlètes ayant gagnés au moins une médaille. Ordonner par nombre de
médailles décroissant. Aide : 2 tables seulement sont nécessaires.
Résultats (24 lignes) : (NOR, 24,19) ; (NED,22,15) ; ...

In [121]:
query="""
select a.codepays , count(*) , count(distinct a.aid)
from athlete a , rangindividuel r
where a.aid = r.aid and r.rang <= 3
group by a.codepays
order by count(*) desc
"""
execute(connection,query,show=True)

24 rows, display only 10 rows
|codepays|?column?|?column?|
|---------------------------
|NOR     |24      |19      |
|NED     |22      |15      |
|USA     |20      |20      |
|RUS     |19      |16      |
|CAN     |17      |16      |
|FRA     |14      |12      |
|AUT     |13      |11      |
|GER     |12      |11      |
|SLO     |10      |5       |
|CHN     |8       |8       |
|SUI     |8       |7       |


<cursor object at 0x7f818ab16b60; closed: -1>

## **Q12.** 
Pour chaque pays et sport, le code du pays, le sid du sport, le nombre de médailles en
épreuve individuelle gagnées, le nombre d'athlètes ayant gagnés au moins une médaille,
ordonner d'abord par code pays, puis par nombre de médailles décroissant.
Résultats (84 lignes) : (AUS,12,2,2); (AUS,15,1,1);(AUT,13,9,7);(AUT,15,2,2);...

In [129]:
query="""
select a.codepays , s.sid , count(*) , count(distinct a.aid)
from athlete a, sport s, rangindividuel r, epreuve e
where a.aid = r.aid and e.epid = r.epid and s.sid = e.sid  and r.rang <= 3
group by a.codepays , s.sid
order by a.codepays , count(*) desc 
"""
execute(connection,query,show=True)

84 rows, display only 10 rows
|codepays|sid|?column?|?column?|
|-------------------------------
|AUS     |12 |2       |2       |
|AUS     |15 |1       |1       |
|AUT     |13 |9       |7       |
|AUT     |15 |2       |2       |
|AUT     |1  |1       |1       |
|AUT     |10 |1       |1       |
|BLR     |1  |4       |1       |
|BLR     |12 |2       |2       |
|CAN     |12 |9       |9       |
|CAN     |8  |2       |1       |
|CAN     |9  |2       |2       |


<cursor object at 0x7f818ab171f0; closed: -1>

<u>**Partitionnement avec « group by / having »**</u>

## **Q13.a)** 
L’eqid de la ou des équipes qui sont composées d'exactement 10 athlètes. Résultat (1 ligne) : 226

In [130]:
query="""
select e.*
from equipe e , athletesequipe aq
where e.eqid = aq.eqid 
group by e.eqid
having count(*) = 10
"""
execute(connection,query,show=True)

1 rows, display only 10 rows
|eqid|codepays|
|--------------
|226 |RUS     |


<cursor object at 0x7f818ab172e0; closed: -1>

## **Q13.b)** 
L’eqid de la ou des équipes qui sont composées du plus d’athlètes pour ces JO. Résultats (3 lignes) : 164 ; 165 ; 166

In [143]:
query="""
select a.eqid , count(*) nb
from athletesequipe a
group by a.eqid
having count(*)  = (select max(t.nb)
                    from (select count(*) as nb 
                            from athletesequipe a2
                            group by a2.eqid) t)



"""
execute(connection,query,show=True)

3 rows, display only 10 rows
|eqid|nb|
|--------
|164 |25|
|165 |25|
|166 |25|


<cursor object at 0x7f818ab17c40; closed: -1>

## **Q14.** 
 Le nombre d'épreuves en individuel où il y a eu au moins 100 participants.
 Aide: calculer les epreuves dans une sous-requête dans la clause FROM 
Résultat (1 ligne ) : 2

In [ ]:
query="""


"""
execute(connection,query,show=True)

## **Q15.** 
Le nom des pays qui ont gagné au moins 20 médailles aux épreuves individuelles.
Résultats (3 lignes) : Pays-Bas ; États-Unis ; Norvège

In [ ]:
query="""

"""
execute(connection,query,show=True)

<u>**Division en SQL**</u>

## **Q16.** 
Le sid des sports qui ont des épreuves dans toutes les catégories existantes.
Résultats (3 lignes) : 1 ; 6 ; 7

Principe : pour chaque épreuve, on compte le nombre de catégories, puis on regarde si il est
égal au nombre total de catégories d’épreuves.

In [ ]:
query="""


"""
execute(connection,query,show=True)

## **Q17.** 
Le nom des pays qui ont participé aux épreuves en individuel de tous les sports en individuel. Résultats (3 lignes) : (Russie,12) ; (États-Unis,12) ; (Italie,12)

In [ ]:
query="""


"""
execute(connection,query,show=True)

**<u>BASE FOOFLE</u>**

## Créer les tables et charger les données base_foofle_H2.sql

In [ ]:
schemafile=open("base_foofle_H2.sql",mode="r",encoding='utf-8')
create_schema=schemafile.read()
execute(connection,"drop all objects")
execute(connection, create_schema)
connection.commit()

## **Q1.** 
Pour chaque sponsor, le nom du sponsor, le nombre de joueurs sponsorisés, le montant total
des sommes versées, ordonner par sommes versées décroissantes.
Résultats (7 lignes) : (Adadis,6,2340) ; (Robek,5,1426) ;...

In [ ]:
query="""


"""
execute(connection,query,show=True)

## **Q2.** 
Quelles équipes ont joué au moins dans 3 stades différents ?
Résultats (2 lignes) : Fortiches ; Direkt

In [ ]:
query="""

"""
execute(connection,query,show=True)

## **Q3.** 
Quels sponsors sponsorisent exactement un joueur pour chaque équipe qu'il sponsorise ?
Résultats (2 lignes) : Air Monaco ; Palasse

In [ ]:
query="""


"""
execute(connection,query,show=True)

## **Q4.** 
Quel est le nombre total de kilomètres parcourus par chaque équipe. On suppose qu’après
chaque match, chaque équipe se rend directement au stade où aura lieu son prochain match (d’après la date du match). Aide : il existe 2 matchs ordonnés par leur date pour la même équipe, mais il n’existe pas un 3ième match entre les dates des 2 matchs pour cette équipe. Résultats (4 lignes) : (Fortiches,516) ; (Direkt,671) ; (Piepla,124) ; (Sabar,360)

In [ ]:
query="""


"""
execute(connection,query,show=True)

# Fermer la connexion

In [ ]:
connection.commit() # implicit avec close
connection.close()

In [ ]:
connection